In [ ]:
from datasets import load_dataset

SEED = 42
N_SAMPLES = 200

ds = load_dataset("stanfordnlp/sst2", split="validation")
print("full split size:", len(ds))
print("features:", ds.features)

# fixed, reproducible slice
ds = ds.shuffle(seed=SEED).select(range(N_SAMPLES))
print("sample size:", len(ds))

In [ ]:
label_names = ds.features["label"].names  # confirm which id means positive
print("label names:", label_names)

for ex in ds.select(range(5)):
    print(ex["label"], "|", ex["sentence"])

In [ ]:
print(ds.to_pandas()["label"].value_counts())

In [10]:
import re
import unicodedata

_WS = re.compile(r"\s+")
_URL = re.compile(r"https?://\S+")
_HTML_TAG = re.compile(r"<[^>]+>")
_ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]")

# Undo SST-2-style tokenization: "does n't" -> "doesn't", "movie ." -> "movie."
_DETOK = [
    (re.compile(r"\s+([.,!?;:])"), r"\1"),
    (re.compile(r"\s+(n't|'s|'re|'ve|'ll|'d|'m)\b"), r"\1"),
    (re.compile(r"-lrb-\s*"), "("),
    (re.compile(r"\s*-rrb-"), ")"),
]


def clean_text(text: str, detokenize: bool = False) -> str:
    """Normalize raw text before it goes into a prompt.

    Steps: Unicode NFKC normalization, remove zero-width characters,
    strip HTML tags and URLs, collapse runs of whitespace, trim the ends.
    If detokenize=True, also rejoin pre-tokenized punctuation and
    contractions (useful for SST-2).

    Does not change case or remove words, so the meaning is preserved.
    """
    text = unicodedata.normalize("NFKC", text)
    text = _ZERO_WIDTH.sub("", text)
    text = _HTML_TAG.sub(" ", text)
    text = _URL.sub(" ", text)
    if detokenize:
        for pattern, repl in _DETOK:
            text = pattern.sub(repl, text)
    return _WS.sub(" ", text).strip()

In [11]:
samples = [
    "  it 's a charming   and often affecting journey .  ",
    "this does n't work at all , -lrb- really -rrb- .",
    "Great movie!!\u200b\n\n  Visit https://example.com <br> now",
    "\tTabs\tand\nnewlines  ",
]
for s in samples:
    print(repr(s), "->", repr(clean_text(s, detokenize=True)))

"  it 's a charming   and often affecting journey .  " -> "it's a charming and often affecting journey."
"this does n't work at all , -lrb- really -rrb- ." -> "this doesn't work at all, (really)."
'Great movie!!\u200b\n\n  Visit https://example.com <br> now' -> 'Great movie!! Visit now'
'\tTabs\tand\nnewlines  ' -> 'Tabs and newlines'


In [13]:
for ex in ds.select(range(5)):
    print(repr(ex["sentence"]))
    print(repr(clean_text(ex["sentence"], detokenize=True)))
    print()

'it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right . '
'it gets onto the screen just about as much of the novella as one could reasonably expect, and is engrossing and moving in its own right.'

'my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovingly dished out humor . '
'my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovingly dished out humor.'

"for the most part , director anne-sophie birot 's first feature is a sensitive , extraordinarily well-acted drama . "
"for the most part, director anne-sophie birot's first feature is a sensitive, extraordinarily well-acted drama."

"cq 's reflection of artists and the love of cinema-and-self suggests nothing less than a new voice that deserves to be considered as a possible successor to the best european directors . "
"cq's reflection of artists and the love of cinema-and-self sug